# Annotation: VEP + CADD - significant HARs, EUR WGS

## Description

This notebook functionally annotates the variants identified in the **HARs with convergent signal** (burden + case-control) in the EUR WGS ancestry.

## Import libraries

In [ ]:
import os
import gzip
import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path

print('Libraries OK')

## Paths and definition of the 4 HARs

In [ ]:
DIR_WSPS   = '/home/jupyter/workspace/ws_files'
DIR_NOVA   = f'{DIR_WSPS}/Novalis_v3_R12'
DIR_RESU   = f'{DIR_NOVA}/Results_union'
DIR_ANNOT  = f'{DIR_NOVA}/Annotation'
os.makedirs(DIR_ANNOT, exist_ok=True)

CC_FILE     = f'{DIR_RESU}/CaseControl_EUR_WGS_FDRv3.tsv'
BURDEN_FILE = f'{DIR_RESU}/ALL_BURDEN_FDR.tsv'

# The 4 HARs with convergent burden + case-control signal
HARS = {
    'cluster02482-HAQER0382': {'chr': '2',  'start': 207013,     'end': 207816,
                                'label': 'HAQER0382', 'burden_p': 6.27e-17, 'kernel': 'SKAT maf03'},
    'cluster04096-HAQER0441': {'chr': '4',  'start': 189810407,  'end': 189811184,
                                'label': 'HAQER0441', 'burden_p': 3.56e-8,  'kernel': 'SKAT-O maf03'},
    'cluster01132-HAQER1191': {'chr': '12', 'start': 124438771,  'end': 124439391,
                                'label': 'HAQER1191', 'burden_p': 1.28e-7,  'kernel': 'SKAT maf03'},
    'cluster05013-HAQER1329': {'chr': '7',  'start': 158098969,  'end': 158099663,
                                'label': 'HAQER1329', 'burden_p': 3.56e-9,  'kernel': 'SKAT-O maf01'},
}

# Thresholds
BONF      = 0.05 / 5915   # 8.45e-6
FDR_THRESH = 0.05
ANC, DS   = 'EUR', 'WGS'

print('Paths configured.')
print(f'  Case-control results: {CC_FILE}')
print(f'  Burden results:       {BURDEN_FILE}')
print(f'  Annotation output:    {DIR_ANNOT}')

## Extract Tier 1 variants

In [ ]:
# Load case-control results
cc = pd.read_csv(CC_FILE, sep='\t', low_memory=False)
for col in ('P', 'FDR_BH', 'OR', 'L95', 'U95', 'A1_FREQ'):
    if col in cc.columns:
        cc[col] = pd.to_numeric(cc[col], errors='coerce')

# Parse REF/ALT from ID if not present as columns
if 'REF' not in cc.columns or cc['REF'].isna().all():
    split_id = cc['ID'].astype(str).str.split(':', expand=True)
    if split_id.shape[1] >= 4:
        cc['REF'] = split_id.iloc[:, -2].values
        cc['ALT'] = split_id.iloc[:, -1].values

# Normalize the CHROM column
chrom_col = '#CHROM' if '#CHROM' in cc.columns else 'CHROM'
cc['CHROM_clean'] = cc[chrom_col].astype(str).str.replace('chr','', regex=False)

# Filter to only the 4 HARs of interest
har_ids = list(HARS.keys())
cc_hars = cc[cc['HAR'].isin(har_ids)].copy()
print(f'Case-control variants in the 4 HARs: {len(cc_hars)}')

# Tier 1: significant at FDR < 0.05
tier1 = cc_hars[cc_hars['FDR_BH'] < FDR_THRESH].copy()
print(f'Tier 1 (FDR < 0.05):        {len(tier1)} variants')

# Add the HAR label
tier1['HAR_label'] = tier1['HAR'].map({k: v['label'] for k,v in HARS.items()})
tier1['sig_tier']  = tier1['P'].apply(lambda p: 'Bonf+FDR' if p < BONF else 'FDR')

tier1[['HAR_label','CHROM_clean','POS','REF','ALT','OR','L95','U95','P','FDR_BH','sig_tier']]

## Prepare VEP input


In [ ]:
def build_vep_input(df, chrom_col='CHROM', pos_col='POS', ref_col='REF', alt_col='ALT'):
    """
    Convert a variants DataFrame to the VEP REST API input format.
    Returns a list of strings: 'CHR POS POS REF/ALT 1'
    VEP expects 1-based coordinates, strand=1 for forward.
    """
    records = []
    for _, row in df.iterrows():
        chrom = str(row[chrom_col]).replace('chr','')
        pos   = int(row[pos_col])
        ref   = str(row[ref_col])
        alt   = str(row[alt_col])

        # For deletions: start = pos+1, end = pos+len(ref)-1.
        # VEP expects the deleted bases on the REF side of the allele string
        # (deleted/-); the reversed form (-/alt) makes VEP return no result
        # for deletions (e.g. GGA>G needs to be sent as 'GA/-', not '-/G').
        if len(ref) > len(alt):   # deletion
            vep_start  = pos + 1
            vep_end    = pos + len(ref) - 1
            deleted    = ref[len(alt):]   # deleted bases (anchor stripped)
            allele_str = f'{deleted}/-'
        elif len(ref) < len(alt): # insertion
            vep_start = pos
            vep_end   = pos + 1
            allele_str = f'{ref}/{alt}'
        else:                      # SNV
            vep_start = pos
            vep_end   = pos
            allele_str = f'{ref}/{alt}'

        records.append(f'{chrom} {vep_start} {vep_end} {allele_str} 1')
    return records

# Build inputs
vep_input_t1 = build_vep_input(tier1, chrom_col='CHROM_clean')

print(f'Tier 1 variants for VEP: {len(vep_input_t1)}')
print()
print('Tier 1 examples:')
for v in vep_input_t1[:5]:
    print(' ', v)

## VEP annotation via REST API


In [ ]:
VEP_REST_URL = 'https://rest.ensembl.org/vep/human/region'
HEADERS      = {'Content-Type': 'application/json', 'Accept': 'application/json'}
BATCH_SIZE   = 200   # max 300 per request; using 200 to be safe

def run_vep_rest(variant_list, batch_size=BATCH_SIZE, sleep_between=1.0):
    """
    Call the VEP REST API in batches.
    Returns a list of JSON results (one per variant).
    """
    all_results = []
    batches = [variant_list[i:i+batch_size] for i in range(0, len(variant_list), batch_size)]

    for b_idx, batch in enumerate(batches):
        print(f'  Batch {b_idx+1}/{len(batches)} ({len(batch)} variants)...', end=' ')
        payload = json.dumps({'variants': batch})

        for attempt in range(3):  # retries
            resp = requests.post(VEP_REST_URL, headers=HEADERS, data=payload, timeout=120)
            if resp.status_code == 200:
                data = resp.json()
                all_results.extend(data)
                print(f'OK ({len(data)} results)')
                break
            elif resp.status_code == 429:  # rate limit
                wait = int(resp.headers.get('Retry-After', 5))
                print(f'Rate limit, waiting {wait}s...')
                time.sleep(wait)
            else:
                print(f'ERROR {resp.status_code}: {resp.text[:200]}')
                time.sleep(5)

        if b_idx < len(batches) - 1:
            time.sleep(sleep_between)

    return all_results


def parse_vep_result(result):
    """
    Extract the most relevant fields from a VEP JSON result.
    For non-coding variants, regulatory features and TF motifs are what matters.
    """
    # Variant identifier
    # Normalized key: use VEP's 'input' field (guarantees the same format as add_vep_key)
    _inp   = result.get('input', '')
    _parts = _inp.split()
    var_id = f"{_parts[0]}_{_parts[1]}_{_parts[3]}" if len(_parts) >= 4 else result.get('id', _inp)

    # Overall most severe consequence
    most_severe = result.get('most_severe_consequence', '.')

    # Known variants (rsID, ClinVar)
    colocated = result.get('colocated_variants', [])
    rsid      = next((c['id'] for c in colocated if c.get('id','').startswith('rs')), '.')
    clin_sig  = ';'.join(set(
        cs for c in colocated for cs in c.get('clin_sig', [])
    )) or '.'
    gnomad_af = next(
        (c.get('gnomade_af', c.get('gnomad_af', None))
         for c in colocated if 'gnomade_af' in c or 'gnomad_af' in c),
        None
    )

    # Transcript consequences (nearby genes)
    tc         = result.get('transcript_consequences', [])
    genes      = ';'.join(sorted(set(t.get('gene_symbol','') for t in tc if t.get('gene_symbol')))) or '.'
    tx_conseq  = ';'.join(sorted(set(
        c for t in tc for c in t.get('consequence_terms',[])
    ))) or '.'

    # Regulatory feature consequences (the most important part for HARs)
    reg   = result.get('regulatory_feature_consequences', [])
    reg_biotypes  = ';'.join(sorted(set(r.get('biotype','') for r in reg if r.get('biotype')))) or '.'
    reg_conseqs   = ';'.join(sorted(set(
        c for r in reg for c in r.get('consequence_terms',[])
    ))) or '.'
    reg_feat_ids  = ';'.join(sorted(set(r.get('regulatory_feature_id','') for r in reg if r.get('regulatory_feature_id')))) or '.'
    reg_impact    = ';'.join(sorted(set(r.get('impact','') for r in reg if r.get('impact')))) or '.'

    # TF motif consequences (change in binding affinity)
    motifs        = result.get('motif_feature_consequences', [])
    tf_names      = ';'.join(sorted(set(m.get('motif_name','') for m in motifs if m.get('motif_name')))) or '.'
    tf_score_chg  = ';'.join(
        f"{m.get('motif_name','')}:{m.get('motif_score_change',0):+.3f}"
        for m in motifs if m.get('motif_score_change') is not None
    ) or '.'
    tf_high_inf   = ';'.join(
        m.get('motif_name','') for m in motifs
        if m.get('high_inf_pos') == 'Y'
    ) or '.'  # TFs where it falls on a high-information position (most relevant)

    return {
        'vep_input':       var_id,
        'most_severe':     most_severe,
        'rsID':            rsid,
        'clin_sig':        clin_sig,
        'gnomAD_AF':       gnomad_af,
        'nearest_genes':   genes,
        'transcript_conseq': tx_conseq,
        'reg_biotypes':    reg_biotypes,
        'reg_conseqs':     reg_conseqs,
        'reg_feat_ids':    reg_feat_ids,
        'reg_impact':      reg_impact,
        'TF_names':        tf_names,
        'TF_score_changes': tf_score_chg,
        'TF_high_inf_pos': tf_high_inf,
        '_raw':            result,   # full result, for inspection
    }

print('VEP functions defined.')

### Run VEP - Tier 1 (case-control significant variants)

In [ ]:
print('=== VEP Tier 1 ===')
vep_raw_t1 = run_vep_rest(vep_input_t1)

# Parse results
vep_parsed_t1 = [parse_vep_result(r) for r in vep_raw_t1]
vep_df_t1     = pd.DataFrame(vep_parsed_t1)

# Save the raw JSON just in case
raw_path = f'{DIR_ANNOT}/vep_raw_tier1.json'
with open(raw_path, 'w') as fh:
    json.dump(vep_raw_t1, fh, indent=2)
print(f'\nRaw JSON saved: {raw_path}')
print(f'Parsed results: {len(vep_df_t1)} variants')
vep_df_t1.drop(columns='_raw').head()

## Build a variant key for the merge


In [ ]:
def add_vep_key(df, chrom_col='CHROM', pos_col='POS', ref_col='REF', alt_col='ALT'):
    """Generate the same key that build_vep_input generates, for the merge."""
    keys = []
    for _, row in df.iterrows():
        chrom = str(row[chrom_col]).replace('chr','')
        pos   = int(row[pos_col])
        ref   = str(row[ref_col])
        alt   = str(row[alt_col])

        if len(ref) > len(alt):
            vep_start = pos + 1
            vep_end   = pos + len(ref) - 1
            deleted    = ref[len(alt):]   # same fix as build_vep_input
            allele_str = f'{deleted}/-'
        elif len(ref) < len(alt):
            vep_start = pos
            vep_end   = pos + 1
            allele_str = f'{ref}/{alt}'
        else:
            vep_start = pos
            vep_end   = pos
            allele_str = f'{ref}/{alt}'

        keys.append(f'{chrom}_{vep_start}_{allele_str}')
    return keys

# Add key to tier1
tier1 = tier1.copy()
tier1['vep_key'] = add_vep_key(tier1, chrom_col='CHROM_clean')

# Add key to vep_df_t1 (the 'vep_input' field already holds the variant as VEP processed it)
# VEP returns the original 'input', we use it for the merge
vep_df_t1['vep_key'] = vep_df_t1['vep_input'].str.strip()

print('Keys generated:')
print('  tier1:', tier1['vep_key'].tolist()[:3])
print('  vep  :', vep_df_t1['vep_key'].tolist()[:3])

## Final integrated table

Merges the case-control statistics with the VEP annotation.

In [ ]:
# Merge on vep_key
merged_t1 = tier1.merge(
    vep_df_t1.drop(columns=['_raw'], errors='ignore'),
    left_on='vep_key', right_on='vep_key',
    how='left'
)

# Add burden statistics
burden = pd.read_csv(BURDEN_FILE, sep='\t')
burden_sig = burden[
    (burden['ANCESTRY'] == ANC) & (burden['DATASET'] == DS) &
    (burden['GENE'].isin(har_ids))
].copy()
# One summary row per HAR (best kernel = lowest P)
burden_best = (burden_sig
    .sort_values('Pvalue')
    .groupby('GENE')
    .first()
    .reset_index()
    [['GENE','KERNEL','CLASS','Pvalue','FDR_BH']]
    .rename(columns={'GENE':'HAR','KERNEL':'burden_kernel','CLASS':'burden_class',
                     'Pvalue':'burden_p','FDR_BH':'burden_fdr'})
)
merged_t1 = merged_t1.merge(burden_best, on='HAR', how='left')

# Final columns for the table
cols_final = [
    'HAR_label','CHROM_clean','POS','REF','ALT',
    'sig_tier','OR','L95','U95','P','FDR_BH','A1_FREQ',
    'burden_kernel','burden_class','burden_p','burden_fdr',
    'rsID','gnomAD_AF','clin_sig',
    'nearest_genes','most_severe',
    'reg_biotypes','reg_conseqs','reg_impact',
    'TF_names','TF_score_changes','TF_high_inf_pos',
    'reg_feat_ids',
]
cols_present = [c for c in cols_final if c in merged_t1.columns]
final_table = merged_t1[cols_present].copy()

print(f'Final Tier 1 table: {len(final_table)} variants')
final_table

## Save results

In [ ]:
# Tier 1 table
out_t1 = f'{DIR_ANNOT}/VEP_annotation_tier1_EUR_WGS.tsv'
final_table.to_csv(out_t1, sep='\t', index=False)
print(f'Tier 1 saved: {out_t1}')

## Interpretive summary per HAR

In [ ]:
print('=' * 70)
print('VEP SUMMARY - Tier 1 variants per HAR')
print('=' * 70)

for har_label in ['HAQER0382','HAQER0441','HAQER1191','HAQER1329']:
    sub = final_table[final_table['HAR_label'] == har_label]
    if sub.empty:
        print(f'\n{har_label}: no Tier 1 variants')
        continue

    print(f'\n--- {har_label} ({len(sub)} variant(s)) ---')
    for _, row in sub.iterrows():
        direction = 'protective' if float(row.get('OR', 1)) < 1 else 'risk'
        print(f"  chr{row['CHROM_clean']}:{row['POS']} {row['REF']}>{row['ALT']}")
        print(f"    OR={row.get('OR','?'):.3f}  P={row.get('P','?'):.2e}  [{row.get('sig_tier','?')}]  {direction}")
        print(f"    rsID: {row.get('rsID','.')}  |  gnomAD_AF: {row.get('gnomAD_AF','.')}")
        print(f"    Most severe consequence: {row.get('most_severe','.')}")
        print(f"    Regulatory features: {row.get('reg_biotypes','.')}")
        print(f"    Reg. impact:         {row.get('reg_impact','.')}")
        print(f"    Nearby genes:        {row.get('nearest_genes','.')}")
        print(f"    Affected TFs:        {row.get('TF_names','.')}")
        print(f"    TF score changes:    {row.get('TF_score_changes','.')}")
        print(f"    TF high-inf pos:     {row.get('TF_high_inf_pos','.')}")

print('\n' + '=' * 70)

## CADD annotation (deleteriousness prediction)

In [ ]:
CADD_BASE = 'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7'

def get_cadd_scores(df, chrom_col='CHROM_clean', pos_col='POS',
                    ref_col='REF', alt_col='ALT'):
    """
    Query the CADD API v1.7 (GRCh38) one variant at a time.
    Only returns scores for precomputed SNVs; indels may come back as None.
    """
    scores = {}
    for _, row in df.iterrows():
        chrom = str(row[chrom_col])
        pos   = int(row[pos_col])
        ref   = str(row[ref_col])
        alt   = str(row[alt_col])
        url   = f'{CADD_BASE}/{chrom}:{pos}_{ref}_{alt}'
        try:
            resp = requests.get(url, timeout=20)
            if resp.ok:
                data = resp.json()
                if isinstance(data, list) and data:
                    phred = data[0].get('PHRED', None)
                    scores[(chrom, pos, ref, alt)] = phred
                    print(f'  chr{chrom}:{pos} {ref}>{alt} -> CADD_phred = {phred}')
                else:
                    print(f'  chr{chrom}:{pos} {ref}>{alt} -> not precomputed')
            else:
                print(f'  chr{chrom}:{pos} HTTP {resp.status_code}')
        except Exception as e:
            print(f'  Error: {e}')
        time.sleep(0.5)   # respect the rate limit
    return scores

print('Querying CADD v1.7 (GRCh38)...')
cadd_scores = get_cadd_scores(final_table)

final_table['CADD_phred'] = final_table.apply(
    lambda r: cadd_scores.get(
        (str(r['CHROM_clean']), int(r['POS']), str(r['REF']), str(r['ALT'])), None
    ), axis=1
)

print('\n', final_table[['HAR_label','POS','REF','ALT','OR','CADD_phred']].to_string())
final_table.to_csv(f'{DIR_ANNOT}/VEP_annotation_tier1_EUR_WGS.tsv', sep='\t', index=False)
print('\nTSV updated with CADD_phred')